# Intro til Machine Learning — 2: Aktiveringsfunktioner

I notebook 1 var ReLU og sigmoid bare "krølning mellem lagene". Nu skal de i rampelyset,
for aktiveringsfunktionerne er dét, der overhovedet gør neurale netværk til andet end
lineær regression i festtøj.

I denne notebook:
1. **De fem store**: Sigmoid, Tanh, ReLU, Leaky ReLU og Softmax — formler, grafer og gradienter (afsnit 1)
2. **Aktiveringer i kamp**: vi træner netværk på måne-formede data og SER forskellen (afsnit 2)

> **Om opgaverne:** Der er med vilje flere opgaver, end I kan nå — ingen forventes at nå alt. Opgaver mærket **(ekstra)** er til jer, der er foran; opgaver mærket **(find fejlen)** har en bevidst fejl, som I skal finde og rette (så en fejl dér er meningen).

## Setup

# 3: Træningsloopet — er den Pokémon legendarisk?

Tid til at samle det hele! Missionen: træn et netværk, der ud fra de seks kamp-stats
afgør, om en Pokémon er **legendarisk**. Det er **binær klassifikation** — svaret er
0 (almindelig) eller 1 (legendarisk) — så opskriften er:

- Output-laget skal give **én** sandsynlighed → 1 output-neuron med **sigmoid** til sidst.
- Tabsfunktionen skal måle "hvor forkert er en sandsynlighed?" → **`nn.BCELoss`**
 (*binary cross entropy* — den straffer hårdt, når modellen er *selvsikker og forkert*).

## Trin 1: Train/test-split — eksamensreglen

Vi deler dataene i **træningssæt** (80 %) og **testsæt** (20 %). Modellen må KUN lære af
træningssættet — testsættet er gemt væk til den afsluttende eksamen. At måle på data,
modellen har trænet på, svarer til at give eleverne facitlisten med til eksamen: flotte
karakterer, nul viden.

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)          # standardisering, som altid
y_np = df["Legendary"].astype(int).values.astype("float32")   # True/False → 1.0/0.0

X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

print("træning:", X_train.shape, "| test:", X_test.shape)
print("andel legendariske i træningssættet:", round(y_train.mean().item(), 3))

## Trin 2: Modellen

6 stats ind → 16 skjulte neuroner (ReLU) → 1 sandsynlighed ud (sigmoid):

In [ ]:
class LegendarySpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.sigmoid(x)      # output bliver en sandsynlighed mellem 0 og 1
        return x

model = LegendarySpotter()
print(model)

## Trin 3: Træningsloopet

Præcis samme rytme som i regressionsforløbet — læg mærke til de fem nummererede trin, for de er
**altid** de samme. Ny ven: optimizeren **Adam**, en smartere udgave af SGD, der løbende
justerer skridtlængden selv. Den er standardvalget i praksis:

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []

for epoch in range(500):
    # 1. Nulstil gradienter
    optimizer.zero_grad()
    # 2. Forward: modellens gæt for alle trænings-Pokémon
    y_hat = model(X_train).squeeze()
    # 3. Regn tabet: hvor forkerte er gættene?
    loss = loss_fn(y_hat, y_train)
    # 4. Backward: find alle gradienter
    loss.backward()
    # 5. Tag ét skridt ned ad bakken
    optimizer.step()

    loss_history.append(loss.item())   # .item(): træk TALLET ud af tensoren

print("sluttab:", round(loss_history[-1], 4))

plt.plot(loss_history)
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.title("Tabskurven — den skal falde og flade ud")
plt.show()

## Trin 4: Eksamen — hvor god er den på testsættet?

Modellen svarer med sandsynligheder. Vi runder af ved 0,5 ("tror du mest ja eller mest
nej?") og sammenligner med de rigtige svar. Andelen af rigtige gæt kaldes **accuracy** —
og husk tricket fra opgave 3.8: gennemsnittet af en True/False-række er netop andelen af True:

In [ ]:
with torch.no_grad():
    probabilities = model(X_test).squeeze()

pred = (probabilities > 0.5).float()          # sandsynlighed → 0 eller 1
accuracy = (pred == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

Pæn høj accuracy!...eller hvad?

## Fælden: den dovne model

Kun ca. 8 % af alle Pokémon er legendariske. En "model", der ALTID svarer "ikke
legendarisk" — uden at kigge på så meget som én stat — ville altså ramme rigtigt ca. 92 %
af gangene. **Høj accuracy er billig, når klasserne er skæve!** Det tjekker vi i opgave
10.6, og i opgave 3.5-10.8 finder vi ud af, hvad modellen egentlig har lært.

### Opgaver

##### Opgave 3.1
Skabelonen nedenfor er træningsloopet med fem huller — ét pr. nummereret trin. Udfyld dem
(kig IKKE i cellen ovenfor... okay, kig lidt hvis I sidder fast ).

In [ ]:
model_a = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=0.01)

for epoch in range(500):
    # 1. Nulstil gradienter
    ...
    # 2. Forward: modellens gæt
    y_hat = ...
    # 3. Regn tabet
    loss = ...
    # 4. Backward: find gradienterne
    ...
    # 5. Tag ét skridt
    ...

print("sluttab:", round(loss.item(), 4))

##### Opgave 3.2
Skru på `lr` og antal epoker i træningsloopet, og mål test-accuracy hver gang (genbrug
eksamens-cellen). Hvor højt kan I komme — kan I slå 95 %? Notér jeres bedste resultat, og
kør jeres bedste opskrift to gange: får I samme tal?

In [ ]:
model_b = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_b.parameters(), lr=0.01)   # ← skru her
for epoch in range(500):                                       # ← og her
    optimizer.zero_grad()
    y_hat = model_b(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = (model_b(X_test).squeeze() > 0.5).float()
print(f"test-accuracy: {(pred == y_test).float().mean().item():.1%}")

##### Opgave 3.3 (find fejlen)
Loopet nedenfor kører uden fejlbeskeder... men tabet rokker sig ikke ud af stedet, og
accuracy er elendig. Rytmen er kommet i uorden: kig på rækkefølgen af de fem trin og ret
den. (Hvad SKAL der være sket, før `step()` kan tage et fornuftigt skridt? Og hvornår må
man tidligst nulstille?)

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    optimizer.step()          # skridt...?
    loss.backward()            # ...før gradienterne er regnet?
    optimizer.zero_grad()     # ...og så slettes de nye gradienter?!

print("sluttab:", round(loss.item(), 4), "— det rokker sig ikke!")

##### Opgave 3.4 (find fejlen)
Nogen ville gemme tabskurven, men plottet crasher med en fejl om "requires grad". Kig på
linjen med `append` — hvad mangler der? (Tip: hvad gør `.item()`, og hvorfor brugte vi den
i det rigtige loop?)

In [ ]:
model = LegendarySpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_history = []

for epoch in range(200):
    optimizer.zero_grad()
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    loss.backward()
    optimizer.step()
    loss_history.append(loss)      # hmm...

plt.plot(loss_history)
plt.show()

##### Opgave 3.5
Træn tre modeller med læringsraterne **0.001, 0.05 og 1.0** og plot deres tre tabskurver
i samme figur (skabelonen er klar — den mangler bare `label` og `plt.legend()`). Sæt ord
på hver kurve: tålmodig? effektiv? kaotisk?

In [ ]:
for lr in [0.001, 0.05, 1.0]:
    model_e = LegendarySpotter()
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model_e.parameters(), lr=lr)
    history = []
    for epoch in range(300):
        optimizer.zero_grad()
        loss = loss_fn(model_e(X_train).squeeze(), y_train)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    plt.plot(history)   # ← tilføj label=f"lr = {lr}"

# ← tilføj plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 3.6
Den dovne model: beregn accuracy for "modellen", der ALTID gætter "ikke legendarisk"
(altså gæt = 0 for alle), og sammenlign med jeres netværk. Slår jeres netværk overhovedet
den dovne? Og find så ud af det VIGTIGE tal: af de legendariske Pokémon i testsættet —
hvor mange fanger hver "model"?

In [ ]:
lazy_pred = torch.zeros(len(y_test))
lazy_accuracy = (lazy_pred == y_test).float().mean()
print(f"den dovne models accuracy: {lazy_accuracy.item():.1%}")

# jeres netværks accuracy (genbrug fra tidligere):
...

# hvor mange af testsættets legendariske fanger netværket?
# tip: kig kun på rækkerne hvor y_test == 1
...

##### Opgave 3.7
Udfyld eksamens-cellen: sandsynligheder → 0/1-gæt (grænse 0,5) → accuracy.

In [ ]:
with torch.no_grad():
    probabilities = model(X_test).squeeze()

pred = (probabilities > ...).float()
accuracy = (... == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

##### Opgave 3.8
Hvor meget kan netværket nå med kun **2** features? Træn en model, der kun ser `HP` og
`Defense` (skabelonen har allerede skåret X ned — kolonne 0 og 2). Hvad sker der med
accuracy — og vigtigere: hvor mange legendariske fanger den? Prøv bagefter med et andet
feature-par og se, hvor meget PARRET betyder.

In [ ]:
X_train2 = X_train[:, [0, 2]]    # kun HP og Defense — prøv også andre par!
X_test2 = X_test[:, [0, 2]]

class SmallSpotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(..., 16)   # ← hvor mange features ser den nu?
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.layer2(self.activation(self.layer1(x))))

model2 = SmallSpotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.01)
for epoch in range(500):
    optimizer.zero_grad()
    loss = loss_fn(model2(X_train2).squeeze(), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = (model2(X_test2).squeeze() > 0.5).float()
print(f"accuracy med 2 features: {(pred == y_test).float().mean().item():.1%}")
print("fangede legendariske:", int(pred[y_test == 1].sum()), "af", int((y_test == 1).sum()))

##### Opgave 3.9 (ekstra)
To identiske netværk, to identiske loops — men det ene træner på de RÅ stats (uden
standardisering) med SGD, det andet på de standardiserede. Kør cellen og sammenlign.
Payoff for regressionsforløbet! (Bemærk: vi bruger SGD her; prøv bagefter med Adam og se, at den
delvist redder situationen — men stadig ikke helt.)

In [ ]:
X_raw = torch.tensor(df[seks_stats].values.astype("float32"))
X_raw_train, X_raw_test, _, _ = train_test_split(X_raw.numpy(), y_np, test_size=0.2, random_state=42)
X_raw_train = torch.tensor(X_raw_train)

for name, data in [("standardiseret", X_train), ("rå tal", X_raw_train)]:
    torch.manual_seed(0)
    model_t = LegendarySpotter()
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.SGD(model_t.parameters(), lr=0.1)
    for epoch in range(500):
        optimizer.zero_grad()
        loss = loss_fn(model_t(data).squeeze(), y_train)
        loss.backward()
        optimizer.step()
    print(f"{name}: sluttab = {round(loss.item(), 4)}")

##### Opgave 3.10 (ekstra)
Byg klassifikationsopgaven om til **regression**: forudsig `HP` ud fra de fem ANDRE stats.
Tre ting skal ændres i forhold til LegendeSpotter-opskriften: (1) output-aktiveringen
(sigmoid?! til HP-værdier på 20-255?), (2) tabsfunktionen, (3) målingen til sidst (accuracy
giver ikke mening for kommatal — brug fx den gennemsnitlige fejl i HP-point).

In [ ]:
fem = ["Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_hp = df[fem].values.astype("float32")
X_hp = (X_hp - X_hp.mean(axis=0)) / X_hp.std(axis=0)
y_hp = df["HP"].values.astype("float32")

X_hp_train, X_hp_test, y_hp_train, y_hp_test = train_test_split(X_hp, y_hp, test_size=0.2, random_state=42)
X_hp_train, X_hp_test = torch.tensor(X_hp_train), torch.tensor(X_hp_test)
y_hp_train, y_hp_test = torch.tensor(y_hp_train), torch.tensor(y_hp_test)

# byg model + loop her — start fra LegendeSpotter-opskriften og lav de tre ændringer
...

##### Opgave 3.11
Hvorfor snyder man sig selv, hvis man måler accuracy på **træningsdataene**? Og hvad tror
I, der sker med (a) accuracy på træningssættet og (b) accuracy på testsættet, hvis man
lader et STORT netværk træne i ekstremt mange epoker på et LILLE datasæt?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

In [ ]:
# Plottehjælperen fra GitHub (Plan B: upload filen manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, make_blobs

from helpers import plot_decision_boundary

torch.manual_seed(42)
np.random.seed(42)

# 1: De fem store

Hurtig genopfriskning af *hvorfor* (opgave 2.6): lineært efter lineært er stadig lineært.
Uden noget ikke-lineært mellem lagene er selv verdens dybeste netværk bare én stor
`nn.Linear`. Aktiveringsfunktionen er den lille bøjning, der — gentaget gennem mange lag —
lader netværket tegne vilkårligt krøllede sammenhænge.

Her er familien:

| Funktion | Formel | Output-interval | Typisk brug |
|---|---|---|---|
| **Sigmoid** | $\sigma(z) = \dfrac{1}{1+e^{-z}}$ | $(0, 1)$ | output ved **binær klassifikation** |
| **Tanh** | $\tanh(z)$ | $(-1, 1)$ | som sigmoid, men centreret om 0 |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | **standardvalget i skjulte lag** |
| **Leaky ReLU** | $z$ hvis $z>0$, ellers $0{,}01 z$ | $(-\infty, \infty)$ | ReLU med "lækage" — mere om hvorfor nedenfor |
| **Softmax** | $\dfrac{e^{z_i}}{\sum_j e^{z_j}}$ | sandsynligheder, sum = 1 | output ved **klassifikation med flere klasser** |

Lad os *se* dem. Her er en plotteskabelon — `torch.linspace` laver 200 jævnt fordelte
tal fra −5 til 5, og så plotter vi funktionen af dem:

In [ ]:
x = torch.linspace(-5, 5, 200)

plt.plot(x, torch.sigmoid(x))
plt.title("Sigmoid")
plt.xlabel("z")
plt.grid(True)
plt.show()

(De andre hedder `torch.tanh(x)`, `torch.relu(x)` og
`nn.functional.leaky_relu(x)` — dem skal I selv plotte i opgave 1.1.)

## Gradienterne — funktionernes skjulte personlighed

Under træning flyder gradienter **baglæns** gennem netværket (autograd!), og de skal
*igennem* hver aktiveringsfunktion undervejs. Funktionens hældning bestemmer, hvor meget
gradient der slipper igennem — og her gemmer der sig to berømte problemer.

Vi kan bruge autograd til at plotte en funktions gradient uden at kende formlen for den:

In [ ]:
x = torch.linspace(-5, 5, 200, requires_grad=True)
y = torch.sigmoid(x)
y.sum().backward()          # giver gradienten i alle 200 punkter på én gang

plt.plot(x.detach(), y.detach(), label="sigmoid(z)")
plt.plot(x.detach(), x.grad, "--", label="gradient (hældning)")
plt.title("Sigmoid og dens gradient")
plt.legend()
plt.grid(True)
plt.show()

Se på den stiplede kurve: sigmoids gradient er **højst 0,25** — og ude i "halerne"
(store positive/negative z) er den praktisk talt **nul**. Sender man en gradient baglæns
gennem mange sigmoid-lag, ganges den med et lille tal for hvert lag og **forsvinder**:
det berømte *vanishing gradient*-problem. Netværkets forreste lag lærer aldrig noget.
(I mærker det selv i opgave 2.5.)

ReLU har det modsat: hældning præcis 1 for alle positive z (gradienten passerer urørt!) —
men hældning **0** for alle negative. En neuron, der altid får negative input, får aldrig
gradient og kan aldrig lære igen: en **død ReLU**. Derfor opfandt man **Leaky ReLU**,
som lader en lille smule gradient sive igennem på den negative side — et plaster på
problemet.

## Softmax: sandsynlighedsmaskinen

De fire første funktioner arbejder på ét tal ad gangen. **Softmax** er anderledes: den
tager en hel **stribe** tal (ét "point" pr. klasse) og laver dem om til sandsynligheder,
der er positive og summer til 1:

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])            # netværkets rå point for 3 klasser
probabilities = torch.softmax(point, dim=0)

print(probabilities)
print("sum:", probabilities.sum().item())

Klassen med flest point får den største sandsynlighed, men de andre får også en bid —
softmax siger ikke bare "vinderen tager alt", den siger *hvor sikker* den er.

### Opgaver

##### Opgave 1.1
Brug plotteskabelonen til at tegne **alle fire** af de "almindelige" funktioner — sigmoid,
tanh, ReLU og leaky ReLU — i ét 2×2-subplot-grid (genbrug subplot-tricket fra opgave 1.8).
Aflæs for hver: hvad er output-intervallet?

In [ ]:
x = torch.linspace(-5, 5, 200)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
axes[0, 0].plot(x, torch.sigmoid(x))
axes[0, 0].set_title("Sigmoid")
# ... fortsæt med tanh, relu og leaky_relu (nn.functional.leaky_relu(x))
plt.tight_layout()
plt.show()

##### Opgave 1.2
Byg selv sigmoid ud af `torch.exp`: udfyld formlen $\sigma(z) = \dfrac{1}{1+e^{-z}}$ og
tjek, at jeres udgave giver det samme som `torch.sigmoid`.

In [ ]:
def min_sigmoid(z):
    return 1 / (1 + ...)

z = torch.tensor([-2.0, 0.0, 3.0])
print("min:    ", min_sigmoid(z))
print("PyTorch:", torch.sigmoid(z))

##### Opgave 1.3
Byg selv ReLU. Udfyld med **enten** `torch.clamp(z, min=...)` (som "klipper" tal af ved en
grænse) **eller** `torch.where(betingelse, hvis_sand, hvis_falsk)` — og tjek mod PyTorch.

In [ ]:
def min_relu(z):
    return ...

z = torch.tensor([-3.0, -0.5, 0.0, 2.0])
print("min:    ", min_relu(z))
print("PyTorch:", torch.relu(z))

##### Opgave 1.4
Genbrug gradient-plotteskabelonen (fra teorien ovenfor) på sigmoid, og aflæs: hvad er
gradienten cirka ved $z = \pm 5$? Forestil jer nu en gradient, der skal baglæns gennem
**10 sigmoid-lag**, hvor den hver gang ganges med et tal af den størrelse — hvad sker der
med den, og hvorfor er det et problem for læring?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.5
Leaky ReLU har en indstillelig "lækage": `nn.functional.leaky_relu(x, negative_slope=...)`.
Plot den med `negative_slope` på **0.01, 0.1 og 0.5** i samme figur (med labels og legend).
Hvornår holder den op med at ligne ReLU og begynder bare at ligne en ret linje?

In [ ]:
x = torch.linspace(-5, 5, 200)
plt.plot(x, nn.functional.leaky_relu(x, negative_slope=0.01), label="0.01")
# ← tilføj 0.1 og 0.5
plt.legend()
plt.grid(True)
plt.show()

##### Opgave 1.6
Udfyld softmax-kaldet (hvilken `dim`?), og tjek at summen er 1. Gang derefter alle point
med 10 (`point * 10`) og kør igen — hvad sker der med sandsynlighederne, og hvorfor giver
det mening?

In [ ]:
point = torch.tensor([2.0, 1.0, 0.1])
probabilities = torch.softmax(point, dim=...)
print(probabilities, "| sum:", probabilities.sum().item())

##### Opgave 1.7 (find fejlen)
Her er softmax brugt på et batch med ét eksempel og tre klasser — men ALLE
"sandsynligheder" bliver 1.0?! Kig på tensorens shape og på `dim`-argumentet: hvilken
retning bliver der normaliseret langs? Ret `dim`, så det giver mening.

In [ ]:
point = torch.tensor([[2.0, 1.0, 0.1]])     # shape (1, 3): 1 eksempel, 3 klasser
probabilities = torch.softmax(point, dim=0)
print(probabilities)                        # [[1., 1., 1.]] ... det er vist ikke sandsynligheder

##### Opgave 1.8
Vælg aktiveringsfunktion til hver situation — og begrund kort:

(a) de skjulte lag i et netværk,
(b) output-laget ved binær klassifikation (spam/ikke spam),
(c) output-laget ved klassifikation med 10 klasser (håndskrevne cifre!),
(d) output-laget ved regression (forudsig en huspris).

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.9 (ekstra)
Plot **gradienterne** af sigmoid, tanh, ReLU og leaky ReLU i ét samlet plot (genbrug
autograd-tricket — én funktion ad gangen, samme `x`). Hvilken funktion har den "sundeste"
gradient for store positive $z$ — og hvad med for store negative?

In [ ]:
funktioner = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "ReLU": torch.relu,
    "leaky ReLU": nn.functional.leaky_relu,
}

for name, f in funktioner.items():
    x = torch.linspace(-5, 5, 200, requires_grad=True)
    f(x).sum().backward()
    plt.plot(x.detach(), x.grad, label=name)   # x.grad ER gradienten

plt.legend()
plt.title("Gradienter")
plt.grid(True)
plt.show()

# 2: Aktiveringer i kamp — månedata

Nu skal påstandene testes. `make_moons` laver et syntetisk 2D-datasæt: to klasser formet
som halvmåner, der griber ind i hinanden. Det er perfekt til formålet, for med kun 2
features kan vi **tegne alt**, inklusive hvad netværket tænker:

In [ ]:
X_np, y_np = make_moons(n_samples=400, noise=0.2, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolors="black", s=25)
plt.title("To måner — kan et netværk skille dem ad?")
plt.show()

Ingen ret linje kan skille de to måner ad — prøv selv at "tegne" en med øjnene.
Spørgsmålet er, om et netværk kan tegne noget bedre.

Vi genbruger træningsopskriften fra notebook 2 og pakker den i en funktion (så vi ikke
skal skrive de samme 10 linjer 10 gange — det er jo derfor, funktioner findes). Læs den
igennem: det er PRÆCIS de fem trin, I kender:

In [ ]:
# train() er nu i helpers (samme 5-trins loop) — vi importerer den bare
from helpers import train


In [ ]:
class MoonNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.activation = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

model = MoonNet()
history = train(model, X, y)
print("sluttab:", round(history[-1], 4))

Og nu det fede: `plot_beslutningsgraense` (fra hjælpefilen) farver **hele planen**
efter, hvad modellen ville svare i hvert punkt. Den sorte streg er netværkets
**beslutningsgrænse** — skillelinjen mellem "klasse 0" og "klasse 1":

In [ ]:
plot_decision_boundary(model, X, y, title="MaaneNet med ReLU")

Se den kurve! Netværket har selv opfundet en bugtet grænse, der følger månerne.
DET er, hvad aktiveringsfunktioner køber os. Men påstanden var jo, at det ikke kan lade
sig gøre *uden* dem — det tjekker I selv i opgave 2.1.

### Opgaver

##### Opgave 2.1
Klassen nedenfor er MaaneNet **uden aktivering mellem lagene** (sigmoiden til sidst er kun
for at få en sandsynlighed ud). Træn den og plot dens beslutningsgrænse. Sammenlign med
ReLU-udgaven ovenfor: hvad er grænsen reduceret til — og hvorfor kan den ALDRIG lære
månerne, uanset træningstid? (Opgave 2.6 har svaret.)

In [ ]:
class LinearNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)          # ingen aktivering!
        x = self.layer2(x)          # ingen aktivering!
        x = self.sigmoid(self.layer3(x))
        return x

model_lin = LinearNet()
history = train(model_lin, X, y)
print("sluttab:", round(history[-1], 4))
plot_decision_boundary(model_lin, X, y, title="Uden aktivering")

##### Opgave 2.2
Byt ReLU ud med **Sigmoid**, **Tanh** og **LeakyReLU** (én ad gangen — skabelonen tager
aktiveringen som parameter) og plot de fire tabskurver i samme figur. Hvem lærer hurtigst
på månerne?

In [ ]:
class FlexNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.layer2 = nn.Linear(16, 16)
        self.layer3 = nn.Linear(16, 1)
        self.activation = activation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.activation(self.layer2(x))
        x = self.sigmoid(self.layer3(x))
        return x

history = train(FlexNet(nn.ReLU()), X, y)
plt.plot(history, label="ReLU")
# ← tilføj nn.Sigmoid(), nn.Tanh() og nn.LeakyReLU()
plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 2.3
Skru op for støjen i `make_moons` (prøv `noise=0.3`, `0.5` og `0.8`), gentræn og plot
beslutningsgrænsen hver gang. Hvornår "knækker" modellen — og hvordan ser grænsen ud, når
den prøver at lære rent kaos?

In [ ]:
X_np2, y_np2 = make_moons(n_samples=400, noise=0.3, random_state=42)   # ← skru på noise
X2 = torch.tensor(X_np2, dtype=torch.float32)
y2 = torch.tensor(y_np2, dtype=torch.float32)

model_noise = MoonNet()
train(model_noise, X2, y2)
plot_decision_boundary(model_noise, X2, y2, title="noise = 0.3")

##### Opgave 2.4 (find fejlen)
Netværket nedenfor skal lave **regression**: forudsige $y \approx 2x + 1$ (værdier fra 1
til 11). Men forudsigelserne klistrer fast lige over 1, og tabet er enormt. Kig på
`forward` — der er sneget sig et lag ind, som IKKE hører hjemme i en regressionsmodel
(opgave 1.8d!). Fjern det og se forskellen.

In [ ]:
x_reg = torch.linspace(0, 5, 100).reshape(-1, 1)
y_reg = 2 * x_reg.squeeze() + 1 + torch.randn(100) * 0.3

class RegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.layer1(x))
        x = self.sigmoid(self.layer2(x))    # hmm...
        return x

model_reg = RegNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model_reg.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model_reg(x_reg).squeeze(), y_reg)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    plt.scatter(x_reg, y_reg, alpha=0.4)
    plt.plot(x_reg, model_reg(x_reg).detach(), color="crimson", linewidth=2)
plt.title(f"sluttab: {loss.item():.2f}")
plt.show()

##### Opgave 2.5 (ekstra)
Vanishing gradients LIVE: skabelonen bygger et **6-lags** netværk, hvor aktiveringen kan
vælges. Træn én udgave med `nn.Sigmoid()` og én med `nn.ReLU()`, og plot de to tabskurver
sammen. Hvad ser I — og hvad har det med opgave 1.4 at gøre?

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.layer = nn.ModuleList([nn.Linear(2, 16)] +
                                 [nn.Linear(16, 16) for i in range(4)] +
                                 [nn.Linear(16, 1)])
        self.activation = activation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        for layer in self.layer[:-1]:
            x = self.activation(layer(x))
        return self.sigmoid(self.layer[-1](x))

history = train(DeepNet(nn.Sigmoid()), X, y, epochs=2000)
plt.plot(history, label="6 lag, sigmoid overalt")
# ← træn og plot også DybtNet(nn.ReLU())
plt.legend()
plt.xlabel("epoke")
plt.ylabel("tab")
plt.show()

##### Opgave 2.6 (ekstra)
Klassifikation med **3 klasser**: `make_blobs` laver tre klumper. Ved flere klasser skal
netværket give **3 tal ud** (ét point pr. klasse), og tabsfunktionen skal være
`nn.CrossEntropyLoss`. Udfyld de tre huller.

**VIGTIG detalje** (I får brug for den i notebook 4): `nn.CrossEntropyLoss` vil have de
**RÅ point** — den kører selv softmax indeni! Derfor har modellen INGEN
aktiveringsfunktion til sidst. Og målene `y` skal være hele tal (klassenumre), ikke
kommatal.

In [ ]:
X_np3, y_np3 = make_blobs(n_samples=450, centers=3, cluster_std=1.2, random_state=42)
X3 = torch.tensor(X_np3, dtype=torch.float32)
y3 = torch.tensor(y_np3, dtype=torch.long)     # klassenumre 0, 1, 2 — hele tal!

class ThreeClassNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, ...)          # ← hvor mange tal skal ud?

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))   # rå point — ingen softmax her!

model3 = ThreeClassNet()
loss_fn = ...                              # ← tabsfunktionen til flere klasser
optimizer = torch.optim.Adam(model3.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(model3(X3), ...)         # ← hvad skal der sammenlignes med?
    loss.backward()
    optimizer.step()

print("sluttab:", round(loss.item(), 4))
plot_decision_boundary(model3, X3, y3, title="3 klasser")

##### Opgave 2.7
Hvad kan et netværk med kun **ÉN skjult neuron** (2 → 1 → 1)? Træn det på månerne og plot
beslutningsgrænsen. Beskriv, hvad I ser — og forklar hvorfor grænsen ser sådan ud, når
én ReLU-neuron kun kan "knække" planen én gang.

In [ ]:
class MiniNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 1)     # én enkelt, ensom neuron
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(1, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.layer2(self.activation(self.layer1(x))))

model_mini = MiniNet()
train(model_mini, X, y, epochs=2000)
plot_decision_boundary(model_mini, X, y, title="1 skjult neuron")

##### Opgave 2.8
Sigmoid kom først (1980'erne), ReLU tog over (2010'erne). Ud fra alt, hvad I har set i
denne notebook: giv mindst to grunde til, at ReLU i dag er standardvalget i skjulte lag —
og én situation, hvor sigmoid stadig er det rigtige valg.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*